# Notebook 03 — The Spatial Join
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
- Implement a nearest-neighbour spatial join with `scipy.spatial.cKDTree` from scratch
- Merge gridded environmental values onto point locations
- Understand what becomes harder when points move over time

**Time:** ~20 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import sys, pathlib

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'
sys.path.insert(0, str(WORKSHOP_DIR))
from shared_utils import find_nearest_grid_cells

facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')
grid = pd.read_csv(DATA_DIR / 'climate_grid.csv')
print('Loaded.')

---
## ✅ Worked example — Full cKDTree spatial join

Let's see exactly what `find_nearest_grid_cells()` does under the hood.

In [ ]:
# Step 1: drop facilities without coordinates.
fac = facilities.dropna(subset=['latitude', 'longitude']).copy()

# Step 2: build a cKDTree from grid centroids.
#   The tree indexes (lat, lon) pairs so queries return the nearest grid cell.
grid_coords = grid[['lat', 'lon']].values          # shape (n_cells, 2)
tree = cKDTree(grid_coords)

# Step 3: query the tree for each facility.
point_coords = fac[['latitude', 'longitude']].values   # shape (n_facilities, 2)
distances, indices = tree.query(point_coords)           # indices → row in grid

# Step 4: attach grid_id.
fac['grid_id'] = grid['grid_id'].iloc[indices].values

# Step 5: standard pandas merge to get environmental values.
fac_env = fac.merge(
    grid[['grid_id', 'temp_mean_c', 'pm25_ugm3', 'ndvi', 'precipitation_mm']],
    on='grid_id',
    how='left'
)

print(f'Joined table shape: {fac_env.shape}')
fac_env[['facility_id', 'facility_type', 'district', 'latitude', 'longitude',
          'grid_id', 'temp_mean_c', 'pm25_ugm3']].head(8)

In [ ]:
# Quick check: no missing environmental values after the join
assert fac_env['temp_mean_c'].notna().all(), 'Some facilities still have no temperature value — check the join'
print('✅ All facilities have a temperature value after the join.')

# Sanity-check: how far is each facility from its nearest grid cell?
# (distances are in degrees; multiply by ~111 km/degree for approximate km)
print(f'Max nearest-cell distance: {distances.max():.3f}° ≈ {distances.max()*111:.1f} km')
print(f'Mean nearest-cell distance: {distances.mean():.3f}° ≈ {distances.mean()*111:.1f} km')

---
## 📖 Why Euclidean distance on lat/lon is acceptable here

We used Euclidean (straight-line) distance between lat/lon pairs — not a proper geodetic (sphere-surface) distance. This is a simplification. Here's when it is acceptable:

- **Small area:** Zimbabwe spans roughly 7° of latitude and 7° of longitude (~700 km). Over this distance, the distortion from treating lat/lon as a flat plane is less than **0.5%** — far smaller than the 27 km grid cell size.
- **Same grid spacing in both axes:** Our grid uses 0.25° steps in both latitude and longitude. At Zimbabwe's latitude (~19°S), 0.25° of longitude = ~26 km and 0.25° of latitude = ~28 km — close enough that the tree behaves symmetrically.

For continental-scale studies, or grids with very unequal spacing, you would use `haversine` distances or reproject to a local UTM coordinate system first.

---
## 🧪 Exercise 1 — Reproduce the join for urban facilities only  *(3 points)*

Filter `facilities` to only those with `facility_type == 'hospital'`, drop missing coordinates, then perform the full cKDTree join to get `temp_mean_c` and `pm25_ugm3` for each hospital.

Store the result in `hospitals_env` (a DataFrame that includes the joined columns).

In [ ]:
hospitals = facilities[facilities['facility_type'] == 'hospital'].dropna(subset=['latitude', 'longitude']).copy()

# Hint: just like the worked example above — use find_nearest_grid_cells()
# to attach a grid_id to hospitals, then merge with grid to bring in
# temp_mean_c and pm25_ugm3. Store the result in hospitals_env.

hospitals = find_nearest_grid_cells(hospitals, grid)
hospitals_env = hospitals.merge(grid[['grid_id', 'temp_mean_c', 'pm25_ugm3']], on='grid_id', how='left')

print(f'Hospitals with environmental data: {len(hospitals_env)}')
hospitals_env[['facility_id', 'district', 'temp_mean_c', 'pm25_ugm3']]


---
### ✅ Knowledge Check — Spatial joins for facility-level exposure

*No coding here — just check your understanding before moving on.*

**Q1. Why repeat the same nearest-grid-cell join for hospitals only, instead of just filtering the already-joined `fac_env` table from the worked example?**

- **A.** You can't filter an already-joined table
- **B.** Either approach works here — filtering fac_env to hospitals would give the same answer; redoing the join is just practice with the join mechanics
- **C.** Hospitals require a completely different join method than other facility types
- **D.** Filtering would lose the facility_id column

<details>
<summary>Show answer</summary>

**Answer: B** — Once a join is done for all facilities, filtering to a subset (hospitals) afterwards gives the same result — the exercise repeats the join mainly for practice, not because hospitals need special handling.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 🧪 Exercise 2 — Top 5 highest-PM2.5 facilities  *(2 points)*

Using `fac_env` (the full joined table from the worked example), sort by `pm25_ugm3` descending and print the top 5 rows. Store just those 5 rows in a variable called `top5_pm25`.

In [ ]:
# Hint: fac_env.sort_values('pm25_ugm3', ascending=False).head(5)

top5_pm25 = fac_env.sort_values('pm25_ugm3', ascending=False).head(5)

top5_pm25[['facility_id', 'facility_type', 'district', 'pm25_ugm3']]


---
### ✅ Knowledge Check — Ranking facilities by exposure

*No coding here — just check your understanding before moving on.*

**Q1. What is the public-health usefulness of ranking facilities by PM2.5 exposure and looking at just the top 5?**

- **A.** It has no use — rankings of small subsets are statistically meaningless
- **B.** It helps prioritise which facilities/communities might need air-quality monitoring or interventions first, given limited resources
- **C.** It proves that air pollution affects all facilities equally
- **D.** It is only useful for academic publication, not real decisions

<details>
<summary>Show answer</summary>

**Answer: B** — When resources for follow-up monitoring or intervention are limited, ranking by exposure is a practical way to decide where to act first.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 📖 What changes when locations move over time?

So far we have joined a *static* set of points to a *single-date* grid. That is the simplest version of the problem.

In the pregnancy cohort (Notebooks 04–06), both dimensions change:
1. **Each participant's location can change** between study visits — she may report a different village at visit 1, visit 2, and delivery.
2. **Environmental values change every day** — the exposure grid has one row per grid cell per day across ~700 days.

So the join becomes:
```
For each participant, for each day of pregnancy:
    1. Determine which village she was in on that day.
    2. Find the nearest grid cell to that village.
    3. Merge on (grid_id, date) to get that day's environmental values.
```

That is exactly what Notebooks 04 and 06 implement — using the same cKDTree logic you just practiced.

---
## 🏁 Recap

- Build a `cKDTree` on grid coordinates → query with point coordinates → get indices → attach `grid_id` → pandas merge.
- Always verify the join with `notna().all()` — a missing value after a join means something went wrong.
- The same join pattern scales to millions of participant-days, just applied per-day with an additional date filter.

**Next:** Notebook 04 introduces the pregnancy cohort dataset and builds the per-day exposure timeline.